# Train speaker attribution with LoRA

Use the **AppliedTaskByte** kernel. The notebook loads your local Qwen model, freezes base weights, and trains small LoRA adapters in decoder attention and MLP layers. The vision encoder and visual merger remain frozen. The 4-bit base makes this QLoRA.

Before running:
1. Finish OCR for the saved training split.
2. Run `transcript_prep_VL.ipynb` with `SPLIT = "train"`.
3. Run `build_speaker_prompt_Vl.ipynb` with `SPLITS = ("train", "val")`.
4. Run this notebook. Close the baseline inference kernel first to release GPU memory.

Only groups with reliable speaker targets are requested in training examples. All three annotated images remain available for context. UNKNOWN and NARRATION are valid targets. Validation sequences never enter training.

All cells are initially unexecuted. [PEFT QLoRA instructions](https://huggingface.co/docs/peft/main/developer_guides/quantization).


## 1. Settings

Keep the image budget equal to the baseline for the first comparison. Rank 16, learning rate 0.0001, and three epochs are starting settings, not tuned results. Memory use still depends on sequence length; there is no automatic reduction in image quality.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import re
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'TaskData').is_dir() or (p / 'manga_notebook_codebase/TaskData').is_dir())
if not (ROOT / 'TaskData').is_dir():
    ROOT = ROOT / 'manga_notebook_codebase'
ROOT = ROOT.resolve()

MODEL_PATH = Path(r'C:\MODELS\HF\Qwen2.5-VL-3B-Instruct')
PROMPT_DIR = ROOT / 'outputs/speaker_prompts/train'
PREP_DIR = ROOT / 'outputs/transcript_prep_vl/train'
OUTPUT_DIR = ROOT / 'outputs/vl_training/ordered_speaker_lora_v1'
RESUME_FROM = None  # Set to a checkpoint folder to continue an interrupted run.

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28
MAX_SEQUENCE_TOKENS = 6144  # Stop rather than silently truncate pages or answers.
LORA_RANK = 16
LEARNING_RATE = 1e-4
EPOCHS = 3
GRADIENT_ACCUMULATION = 8  # Eight single-sequence batches per optimizer update.
SEED = 42

import sys
import importlib
sys.path.insert(0, str(ROOT / 'src'))
import vl_prompts
importlib.reload(vl_prompts)  # Pick up edits even in an existing kernel.
from vl_prompts import (OUTPUT_FORMAT, ordered_groups, build_speaker_prompt,
                        build_speaker_target, parse_speaker_output, permute_speaker_labels)


## 2. Load training examples

Check saved split membership and source hashes before loading the model. Partially labelled sequences are usable because their prompts request only accepted groups, and their answers contain exactly those groups.


In [2]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()
            if line.strip()]


def file_hash(path):
    # Read large model files in chunks to keep memory use small.
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


examples_path = PROMPT_DIR / 'training_examples.jsonl'
if not examples_path.is_file():
    raise FileNotFoundError('Prepare training transcripts and prompts first; see the first cell.')
summary = json.loads((PROMPT_DIR / 'summary.json').read_text(encoding='utf-8'))
if summary.get('output_format') != OUTPUT_FORMAT:
    raise ValueError('Rerun build_speaker_prompt_Vl.ipynb for ordered speaker targets.')
if summary['split'] != 'train':
    raise ValueError('Expected training prompt outputs')
if file_hash(examples_path) != summary['output_sha256']['training_examples.jsonl']:
    raise ValueError('Training examples changed; regenerate prompts.')
for name, key in [('inputs.jsonl', 'inputs'), ('targets.jsonl', 'targets')]:
    if file_hash(PREP_DIR / name) != summary['source_sha256'][key]:
        raise ValueError('Training preparation changed; regenerate prompts.')
split_path = ROOT / 'outputs/split.json'
if file_hash(split_path) != summary['source_sha256']['split']:
    raise ValueError('Saved split changed; regenerate prompts.')
if file_hash(ROOT / 'src/vl_prompts.py') != summary['shared_prompt_sha256']:
    raise ValueError('Shared prompt code changed; regenerate prompts.')

examples = read_jsonl(examples_path)
saved_split = json.loads(split_path.read_text(encoding='utf-8'))
ids = [example['sequence_id'] for example in examples]
if not ids or len(ids) != len(set(ids)):
    raise ValueError('Training examples must be nonempty with unique sequence IDs')
if not set(ids).issubset(set(saved_split['train'])) or set(ids) & set(saved_split['val']):
    raise ValueError('Training examples contain sequences outside the saved training split')

# Check images and answer IDs while no GPU memory has been allocated.
image_hashes = {}
for example in examples:
    if len(example['images']) != 3:
        raise ValueError('Each training example needs three images')
    # Parse using the saved ordered groups, never a set or alphabetical ID order.
    requested = example['requested_groups']
    answer = parse_speaker_output(example['assistant_response'], requested)
    if not answer or example['prompt'] != build_speaker_prompt(requested):
        raise ValueError('Training prompt or target does not match requested groups')
    if example.get('output_format') != OUTPUT_FORMAT:
        raise ValueError('Old training target format; regenerate prompts')
    for image in example['images']:
        path = Path(image).resolve()
        if not path.is_relative_to(PREP_DIR.resolve()) or not path.is_file():
            raise ValueError(f'Missing or unsafe training image: {path}')
        relative = str(path.relative_to(PREP_DIR.resolve()))
        image_hashes[relative] = file_hash(path)
        if image_hashes[relative] != summary['annotated_image_sha256'][relative]:
            raise ValueError(f'Training image changed: {path}')
print('Training sequences:', len(examples))
print('Supervised groups:', sum(len(e['requested_groups']) for e in examples))

# Resume only checkpoints from this experiment; old JSON runs stay separate.
if RESUME_FROM is not None:
    checkpoint = Path(RESUME_FROM).resolve()
    if checkpoint.parent != OUTPUT_DIR.resolve() or not checkpoint.is_dir():
        raise ValueError('RESUME_FROM must be a checkpoint inside the current OUTPUT_DIR')


Training sequences: 63
Supervised groups: 1277


## 3. Load Qwen and attach decoder LoRA

Select module names explicitly so adapters cannot accidentally enter the vision encoder or visual merger. Print trainable parameters and check that every trainable weight belongs to decoder LoRA.


In [3]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from transformers import Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError('Use the CUDA-enabled AppliedTaskByte kernel')
set_seed(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(
    MODEL_PATH, local_files_only=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'right'

# The base weights are quantized and remain frozen.
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, local_files_only=True, dtype=compute_dtype,
    quantization_config=quantization, device_map={'': 0}, attn_implementation='sdpa')
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False  # Training uses checkpointing instead of generation caching.

# Train attention and MLP adapters in the language decoder only.
suffixes = {'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'}
target_modules = [name for name, _ in model.named_modules()
                  if '.language_model.layers.' in name and name.split('.')[-1] in suffixes]
if not target_modules:
    raise ValueError('No decoder LoRA modules found; inspect the model architecture')
lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05,
    target_modules=target_modules, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
for name, parameter in model.named_parameters():
    if parameter.requires_grad and ('lora_' not in name or '.language_model.layers.' not in name):
        raise ValueError(f'Unexpected trainable parameter: {name}')
model.print_trainable_parameters()


W1003 20:12:20.375000 27064 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,784,556,544 || trainable%: 0.7909


## 4. Prepare a single training batch

Attach the same three PIL images and mask every user/image token. The assistant contains speaker lines plus the chat ending. Each presentation uses one random character-label permutation across the sequence; UNKNOWN and NARRATION stay fixed. The seeded Python RNG is saved by Trainer checkpoints. Arbitrary permutations still produce multiple valid answers under exact-token loss; this augmentation does not make the loss permutation invariant.


In [4]:
from PIL import Image
from qwen_vl_utils import process_vision_info


def collate_example(records):
    if len(records) != 1:
        raise ValueError('This simple collator supports batch size 1')
    example = records[0]
    content = []
    for path in example['images']:
        with Image.open(path) as source:
            image = source.convert('RGB')
        content.append(dict(type='image', image=image,
                            min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS))
    content.append(dict(type='text', text=example['prompt']))
    user_message = dict(role='user', content=content)
    # Shuffle identities once for this entire three-page presentation.
    # Trainer seeds and checkpoints Python's RNG; workers remain zero for replay.
    speakers = list(parse_speaker_output(
        example['assistant_response'], example['requested_groups']).values())
    shuffled = permute_speaker_labels(speakers)
    assistant_text = '\n'.join(shuffled)
    assistant_message = dict(role='assistant', content=assistant_text)

    # The generation prompt is the exact prefix before the assistant answer.
    prompt_text = processor.apply_chat_template(
        [user_message], tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(
        [user_message, assistant_message], tokenize=False, add_generation_prompt=False)
    images, videos = process_vision_info([user_message])
    try:
        prefix = processor(text=[prompt_text], images=images, videos=videos, return_tensors='pt')
        batch = processor(text=[full_text], images=images, videos=videos, return_tensors='pt')
    finally:
        for item in content:
            if item['type'] == 'image':
                item['image'].close()
        for image in images:
            image.close()
    prefix_length = prefix['input_ids'].shape[1]
    if not torch.equal(batch['input_ids'][:, :prefix_length], prefix['input_ids']):
        raise ValueError('Chat-template token prefix differs; refusing incorrect loss masking')
    if batch['input_ids'].shape[1] > MAX_SEQUENCE_TOKENS:
        raise ValueError(f"{example['sequence_id']}: sequence exceeds MAX_SEQUENCE_TOKENS")
    labels = batch['input_ids'].clone()
    labels[:, :prefix_length] = -100  # Ignore all user text and image tokens; supervise speaker lines only.
    labels[batch['attention_mask'] == 0] = -100
    if not (labels != -100).any():
        raise ValueError('No assistant tokens remain for training')
    batch['labels'] = labels
    return batch


# Inspect token counts before starting optimization; this does not train the model.
example_batch = collate_example([examples[0]])
print('Total tokens:', example_batch['input_ids'].shape[1])
print('Supervised answer tokens:', int((example_batch['labels'] != -100).sum()))
print('Example supervised lines:', processor.decode(
    example_batch['labels'][example_batch['labels'] != -100], skip_special_tokens=True))
del example_batch


Total tokens: 4886
Supervised answer tokens: 139
Example supervised lines: char2
char2
char2
char2
char1
char3
char3
char3
char3
char3
char3
char3
char3
char2
char3
char3
char3
char3
char3
char3
char3
char3
char3
char3
char1
char3
char2
char3
char3
char3
char3
char2
char1
char2
char3
char3
char2
char2
char3
char3
char2
char3
char3
char3
char3
char2



## 5. Train and save adapters

Train for the configured epochs without using validation labels. Checkpoints include optimizer state for resuming. The final folder contains LoRA weights and processor files; the original model is unchanged.

If settings or data change, use a new output folder. GPU memory failures should be addressed explicitly before restarting; the notebook never silently changes the image budget.


In [5]:
from math import ceil

# Transformers 5 uses an explicit number of warmup updates.
updates_per_epoch = ceil(len(examples) / GRADIENT_ACCUMULATION)
total_updates = ceil(EPOCHS * updates_per_epoch)
warmup_steps = ceil(0.1 * total_updates)  # Warm up for the first 10% of updates.

# A small effective batch is achieved through gradient accumulation.
arguments = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE, warmup_steps=warmup_steps, lr_scheduler_type='cosine',
    optim='adamw_torch', max_grad_norm=1.0,
    bf16=compute_dtype == torch.bfloat16, fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=1, save_strategy='epoch', save_total_limit=3, eval_strategy='no',
    remove_unused_columns=False, dataloader_num_workers=0,
    report_to='none', seed=SEED, data_seed=SEED)

# Record inputs, settings, model files, and runtime before training.
settings = dict(rank=LORA_RANK, learning_rate=LEARNING_RATE, epochs=EPOCHS,
                warmup_steps=warmup_steps,
                gradient_accumulation=GRADIENT_ACCUMULATION, seed=SEED,
                min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
                max_sequence_tokens=MAX_SEQUENCE_TOKENS, compute_dtype=str(compute_dtype),
                quantization='NF4', trainable_modules=target_modules,
                output_format=OUTPUT_FORMAT, label_permutation='per-presentation Python RNG',
                training_examples_sha256=file_hash(examples_path),
                split_sha256=file_hash(split_path), image_sha256=image_hashes,
                model_path=str(MODEL_PATH),
                model_sha256={path.name: file_hash(path) for path in MODEL_PATH.iterdir()
                              if path.is_file() and path.suffix in {'.json', '.safetensors', '.txt'}},
                packages={name: importlib.metadata.version(name)
                          for name in ['torch', 'transformers', 'peft', 'accelerate',
                                       'bitsandbytes', 'qwen-vl-utils', 'Pillow']},
                gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda)
notebook = json.loads((ROOT / 'notebooks/train_VL.ipynb').read_text(encoding='utf-8'))
code_sources = [cell['source'] for cell in notebook['cells'] if cell['cell_type'] == 'code']
settings['notebook_code_sha256'] = hashlib.sha256(json.dumps(code_sources).encode()).hexdigest()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = OUTPUT_DIR / 'training_metadata.json'
# A failed setup may leave metadata but no actual training outputs.
has_training_outputs = (any(OUTPUT_DIR.glob('checkpoint-*'))
                        or (OUTPUT_DIR / 'adapter').exists()
                        or (OUTPUT_DIR / 'trainer_state.json').exists())
if metadata_path.exists() and has_training_outputs:
    if json.loads(metadata_path.read_text(encoding='utf-8')) != settings:
        raise ValueError('Training settings changed; choose a new OUTPUT_DIR')
    if RESUME_FROM is None:
        raise ValueError('Run already exists; set RESUME_FROM or choose a new OUTPUT_DIR')
metadata_path.write_text(json.dumps(settings, indent=2) + '\n', encoding='utf-8')

trainer = Trainer(model=model, args=arguments, train_dataset=examples,
                  data_collator=collate_example, processing_class=processor)
result = trainer.train(resume_from_checkpoint=RESUME_FROM)

# Save the adapter separately from the original Qwen weights.
adapter_dir = OUTPUT_DIR / 'adapter'
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
trainer.save_state()
trainer.save_metrics('train', result.metrics)
print('Saved LoRA adapter:', adapter_dir)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.35 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Process 12184 has 17179869184.00 GiB memory in use. Process 13736 has 17179869184.00 GiB memory in use. Process 13860 has 17179869184.00 GiB memory in use. Process 2152 has 17179869184.00 GiB memory in use. Process 16988 has 17179869184.00 GiB memory in use. Process 17108 has 17179869184.00 GiB memory in use. Process 21820 has 17179869184.00 GiB memory in use. Process 23852 has 17179869184.00 GiB memory in use. Process 11252 has 17179869184.00 GiB memory in use. Process 13876 has 17179869184.00 GiB memory in use. Process 24904 has 17179869184.00 GiB memory in use. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 11.35 GiB is allocated by PyTorch, and 2.52 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True 

## 6. Evaluate the new adapter

Release the training GPU memory, then use `validation_VL.ipynb`. Set `ADAPTER_PATH` to a checkpoint or `outputs/vl_training/ordered_speaker_lora_v1/adapter`. Choose a new RUN_NAME for each adapter. Start a fresh run for these new targets; old JSON adapters and checkpoints belong to the previous experiment.
